# CVaR optimal vs equal weighted portfolios.

The following optimization are considered:
1. optimal portfolio with maximum C-Sharpe ratio - 'P_Sharpe'
2. optimal portfolio with minimum CVaR - 'P_MinRisk'
3. optimal CVaR portfolio with expected rate of returns (quarterly) of 0.032 - 'P_Risk'
4. optimal CVaR portfolio with risk aversion coefficient of 0.5 - 'P_RiskAverse'
5. optimal CVaR portfolio with same risk as the equal weighted portfolio - 'P_InvNrisk'
6. same as 1 but using the minimization of inverse C-Sharpe ratio - 'P_Sharpe2'

The equal weighted portfolio is named 'P_N'.

The rebalancing schedule is identical for all portfolios.

We start by importing **azapy** and other useful packages.

In [1]:
import azapy as az

print(f"azapy version {az.version()} >= 1.2.0", flush=True)

azapy version 1.2.5 >= 1.2.0


### Collect historical market data

Note the flag `force=False`. The function will attempt first to read the market data from the local directory `mktdir`. If that fails, then it will access the data provider servers *(in this case yahoo)*.

>Make sure that `mktdir` holds a convenient location to save the market data.
>You can inhibit the saving mechanism by setting `save=False` in the call of `az.readMkT` function (_see `readMkT` documentation_ https://azapy.readthedocs.io/en/latest/).

In [2]:
symb = ['GLD', 'TLT', 'XLV', 'VGT', 'VHT']

sdate = "2012-01-01"
edate = 'today'
mktdir = "../MkTdata"

mktdata = az.readMkT(symb, sdate=sdate, edate=edate, file_dir=mktdir)

read GLD data from file
read TLT data from file
read XLV data from file
read VGT data from file
read VHT data from file

Request between 2012-01-03 : 2024-07-03
                    GLD         TLT         XLV         VGT         VHT
source            yahoo       yahoo       yahoo       yahoo       yahoo
force             False       False       False       False       False
save               True        True        True        True        True
file_dir     ../MkTdata  ../MkTdata  ../MkTdata  ../MkTdata  ../MkTdata
file_format         csv         csv         csv         csv         csv
api_key            None        None        None        None        None
nrow               3145        3145        3145        3145        3145
sdate        2012-01-03  2012-01-03  2012-01-03  2012-01-03  2012-01-03
edate        2024-07-03  2024-07-03  2024-07-03  2024-07-03  2024-07-03
error                No          No          No          No          No
extraction time 0.19 s


### Set dispersion measure parameters

Set the parameters for a mCVaR (mixture of 3 CVaR's) dispersion:
- `alpha` the list of CVaR confidence levels,
- `coef` the list of mixture coefficients.

In [3]:
alpha = [0.95, 0.90, 0.85]
coef = [0.1, 0.3, 0.6]
hlength = 3.25
verbose = False

### Define a dictionary of portfolio parameters

The index is the name of the portfolio, and the values are dictionaries of model parameters:
 - 'type' is the portfolio class name, 
 - 'm_param' is a dictionary of parameters required by the corresponding `set_model` function.
 
 We had adopted this rather encrypted method to facilitate an easy call to the model classes.

In [4]:
models = {'P_Sharpe': {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'Sharpe', 'mu0': 0, 'hlength': hlength, 'verbose': verbose}},
          'P_Risk': {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'Risk', 'mu': 0.035, 'hlength': hlength, 'verbose': verbose}},
          'P_MinRisk': {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'MinRisk', 'hlength': hlength, 'verbose': verbose}},
          'P_InvNrisk': {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'InvNrisk', 'hlength': hlength, 'verbose': verbose}},
          'P_RiskAverse': {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'RiskAverse', 'aversion':0.5, 'hlength': hlength, 'verbose': verbose}},
          'P_Diverse': {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'Diverse', 'mu': 0.035, 'hlength': hlength, 'verbose': verbose}},
          'P_MaxDiverse' : {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'MaxDiverse', 'hlength': hlength, 'verbose': verbose}},
          'P_InvNdiverse' : {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'InvNdiverse', 'hlength': hlength, 'verbose': verbose}},
          'P_InvNdrr' : {'type': 'Port_CVaR', 'm_param': {'alpha': alpha, 'coef': coef, 'rtype': 'InvNdrr', 'hlength': hlength, 'verbose': verbose}},
          'P_N': {'type': 'Port_ConstW', 'm_param': {'ww': None}}}

### Main computation loop

The portfolios time-series are stored in a list while the actual model classes are stored in a dictionary with the index given by the portfolio name. They can be interrogated later for individual computed results.

In [5]:
port = []
pp = {}
for key, val in models.items():
    ppz = getattr(az, val['type'])
    pp_ = ppz(mktdata, pname=key)
    pp[key] = pp_
    port_ = pp_.set_model(**val['m_param'])
    port.append(port_)

### Build a comparison environment 

Using the list of individual portfolio time-series, `port`, we can set a `Port_Simple` class to facilitate the visual and numerical comparisons. We are interested in comparing the components of this portfolio of portfolios and ignore their aggregated time-series. 

>Note the call to `set_model` method that is a must.

>Observation: `Port_Simple` is the class that supports the back testing of "Buy and Hold" portfolio (_see its documentation_).
It also can be used as a tool to compare the performance of multiple portfolios. Here we use it in this latter capacity.

In [6]:
ps = az.Port_Simple(port, col='close', pname='ALL')
_ = ps.set_model()

### Visualize the portfolio time-series

We had used the following flags:
- `componly=True` to plot only the initial portfolio time-series without their aggregated portfolio,
- `fancy=True` to use the interactive `plotly` time-series library.

In [7]:
_ = ps.port_view_all(sdate='2000-01-01', componly=True, fancy=True, title="Relative performance")

### Portfolio performance comparisons 

- `RR` is the average annual portfolio rate of returns. 
- `DD` is the maximum drawdown rate.
- `Beta` is the ratio `RR/DD`.
- `DD_date` is the date of maximum drawdown.
- `DD_start` is the date when the maximum drawdown event had started.
- `DD_end` is the date when the maximum drawdown event had ended. If it is `nan` then the drawdown is still in progress (`DD` and `DD_date` are only provisional).

We had used the same flags as before.

In [8]:
ps.port_perf(componly=True, fancy=True)

,RR,DD,RoMaD,DD_date,DD_start,DD_end,DD_days
symbol,,,,,,,
P_Risk,14.41,-19.10,0.754391,2022-09-30,2021-12-30,2023-07-20,567
P_Diverse,15.10,-22.94,0.658201,2022-10-14,2021-12-30,2023-06-13,530
P_InvNrisk,10.82,-21.92,0.493626,2022-09-26,2021-11-18,2024-02-22,826
P_N,10.04,-21.29,0.471754,2022-10-20,2021-12-30,2024-02-07,769
P_MinRisk,8.48,-19.08,0.444473,2022-10-20,2021-12-30,2024-03-04,795
P_InvNdiverse,11.90,-27.08,0.439504,2022-09-26,2021-12-27,2024-07-03,919
P_Sharpe,9.13,-22.41,0.407518,2022-09-27,2021-11-18,2024-06-20,945
P_MaxDiverse,8.37,-22.27,0.375684,2022-10-20,2021-12-30,NaN,916
P_InvNdrr,8.48,-23.37,0.362672,2022-10-20,2021-12-30,NaN,916


### Portfolio annual returns

We had used the same flags as before.

In [9]:
ps.port_annual_returns(withcomp=True, componly=True, fancy=True)

symbol,P_Diverse,P_InvNdiverse,P_InvNdrr,P_InvNrisk,P_MaxDiverse,P_MinRisk,P_N,P_Risk,P_RiskAverse,P_Sharpe
year,,,,,,,,,,
2015,-2.47%,-2.76%,-2.65%,-4.84%,-2.70%,-4.65%,-3.17%,-3.48%,-5.91%,-4.71%
2016,2.89%,3.44%,2.14%,3.65%,3.96%,3.28%,4.65%,4.57%,0.43%,1.43%
2017,33.03%,28.09%,21.09%,27.90%,23.77%,23.42%,20.34%,33.14%,25.53%,25.60%
2018,10.67%,6.72%,6.72%,5.39%,5.07%,5.32%,5.80%,10.86%,4.60%,6.09%
2019,32.78%,34.88%,27.62%,30.07%,28.80%,26.33%,22.23%,32.11%,33.43%,34.66%
2020,22.67%,34.65%,25.17%,29.64%,21.21%,17.55%,26.33%,20.21%,20.92%,18.49%
2021,6.68%,9.38%,6.83%,7.63%,6.55%,6.56%,12.72%,6.33%,6.05%,6.17%
2022,-16.68%,-19.92%,-16.16%,-11.55%,-14.66%,-10.54%,-14.79%,-10.22%,-14.65%,-11.82%
2023,35.29%,12.89%,6.33%,6.76%,5.56%,7.30%,15.03%,21.68%,6.49%,4.81%


### Portfolios monthly returns

We had used the following flags:
- `withcomp = True` to print the portfolio returns,
- `componly = True` to exclude the aggregated portfolio of portfolios,
- `fancy = True` to print the rates in a percent format.

In [10]:
ps.port_monthly_returns(withcomp=True, componly=True, fancy=True)

### Example of a specific portfolio performance inquiry

In [11]:
pp['P_Sharpe'].port_monthly_returns(fancy=True)

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
month,,,,,,,,,,
1,nan%,-3.65%,2.95%,3.78%,6.38%,3.40%,-2.20%,-4.94%,-0.25%,2.78%
2,nan%,0.26%,3.57%,-0.99%,4.89%,-3.78%,-3.97%,0.07%,-4.80%,3.45%
3,nan%,4.16%,1.17%,1.47%,-1.09%,-0.77%,0.16%,1.18%,3.13%,1.91%
4,nan%,-0.23%,1.99%,-0.74%,3.80%,7.81%,3.97%,-6.31%,2.58%,-4.49%
5,nan%,2.21%,3.44%,5.41%,-5.15%,2.60%,3.56%,-2.61%,-3.56%,3.03%
6,-1.91%,2.93%,-2.95%,-2.25%,7.31%,0.04%,-1.93%,-3.15%,3.36%,2.56%
7,3.11%,4.36%,2.36%,1.22%,2.37%,7.45%,3.29%,0.61%,1.36%,-0.32%
8,-7.16%,-0.98%,3.14%,6.43%,1.08%,0.71%,1.26%,-4.34%,-0.93%,nan%
9,-7.55%,0.85%,1.41%,-0.91%,0.56%,-2.58%,-4.29%,-3.65%,-4.46%,nan%


In [12]:
pp['P_Sharpe'].port_annual_returns(withcomp=True, fancy=True)

,P_Sharpe,GLD,TLT,VGT,VHT,XLV
year,,,,,,
2015,-4.71%,-9.77%,5.03%,0.18%,-5.74%,-4.60%
2016,1.43%,8.03%,1.17%,13.77%,-3.21%,-2.76%
2017,25.60%,12.81%,9.18%,37.08%,23.26%,21.77%
2018,6.09%,-1.94%,-1.61%,2.46%,5.58%,6.28%
2019,34.66%,17.86%,14.12%,48.62%,21.87%,20.45%
2020,18.49%,24.81%,18.15%,46.04%,18.29%,13.30%
2021,6.17%,-4.15%,-4.60%,30.45%,20.57%,26.04%
2022,-11.82%,-0.77%,-31.23%,-29.70%,-5.60%,-2.08%
2023,4.81%,12.69%,2.77%,52.66%,2.52%,2.07%
